In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [4]:
class LLMState(TypedDict):
    question: str
    answer: str

graph = StateGraph(LLMState)

In [5]:
def llm_qa(state: LLMState) -> LLMState:
    # extract the question from state

    question = state['question']
    # form a prompt 
    prompt = f"Answer the following question {question}"
    # ask that question to the LLM
    response = model.invoke(prompt)
    answer = str(response.content)
    # update the answer in the state
    state["answer"] = answer

    return state


In [7]:
graph.add_node('llm_qa',llm_qa)

# add edges
graph.add_edge(START,'llm_qa')
graph.add_edge('llm_qa',END)

# compile
workflow = graph.compile()

In [8]:
# execute
initial_state = {'question':'How far is Sun from the earth?'}
final_state = workflow.invoke(initial_state)
print(final_state)

{'question': 'How far is Sun from the earth?', 'answer': "The average distance between the Sun and the Earth is approximately **150 million kilometers (km)** or **93 million miles**.\n\nThis average distance is also defined as **1 Astronomical Unit (AU)**.\n\nIt's important to note that this is an average, as Earth's orbit around the Sun is not a perfect circle but an ellipse. So, the actual distance varies throughout the year:\n\n*   **Perihelion** (closest point): About 147.1 million km (91.4 million miles), occurring around early January.\n*   **Aphelion** (farthest point): About 152.1 million km (94.5 million miles), occurring around early July.\n\nBut for most general purposes, **150 million kilometers (93 million miles) or 1 AU** is the answer."}


In [9]:
model.invoke('How far is moon to earth?').content

"The distance between the Earth and the Moon is not constant because the Moon orbits the Earth in an elliptical path.\n\nHowever, here's a breakdown:\n\n*   **Average distance:** Approximately **384,400 kilometers (238,900 miles)**.\n*   **Closest point (perigee):** About 363,104 km (225,623 miles).\n*   **Farthest point (apogee):** About 405,696 km (252,088 miles).\n\nSo, while the average is often cited, remember it's always changing!"